In [1]:
from scripts.data_loader import get_splitted_df, LabeledFaceDataset, labeled_face_dataset_collate, PairDataset
from scripts.data_loader import reverse_preprocess_input, pair_dataset_collate
from scripts.model import Arcface

In [2]:
from torch.utils.data import DataLoader
import torch.optim as optim
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

### Prepare Train Dataset

In [3]:
ROOT_PATH = "data/faces_webface_112x112"

In [4]:
csv_path = f"{ROOT_PATH}/train.csv"

In [5]:
df_train, df_val = get_splitted_df(csv_path)

In [6]:
len(df_train), len(df_val)

(392498, 98125)

In [7]:
labels_train = set(df_train["label"].unique())
labels_val = set(df_val["label"].unique())

print("Labels missing from validation:", labels_train - labels_val)
print("Labels missing from training:", labels_val - labels_train)

Labels missing from validation: {np.int64(9282)}
Labels missing from training: set()


In [8]:
len(labels_train), len(labels_val)

(10572, 10571)

In [9]:
train_dataset = LabeledFaceDataset(root_path = ROOT_PATH, df=df_train, split="train")
val_dataset = LabeledFaceDataset(root_path = ROOT_PATH, df=df_val, split="val")

In [46]:
train_batch_size = 32
train_num_workers = 4

In [47]:
train_loader = DataLoader(
    train_dataset, 
    shuffle=True, 
    batch_size=train_batch_size, 
    num_workers=train_num_workers, 
    pin_memory=True,
    collate_fn=labeled_face_dataset_collate, 
)

val_loader = DataLoader(
    val_dataset, 
    shuffle=False, 
    batch_size=train_batch_size, 
    num_workers=train_num_workers, 
    pin_memory=True,
    collate_fn=labeled_face_dataset_collate, 
)

In [12]:
# Checking
# images, labels = next(iter(train_loader))
# reverse_preprocess_input(next(iter(train_loader))[0][0])
# reverse_preprocess_input(next(iter(val_loader))[0][0])

### Config

In [13]:
num_classes = 10572
lr = 1e-3
weight_decay = 5e-4
momentum = 0.9

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [43]:
model = Arcface(num_classes=num_classes).to(device)

In [16]:
num_classes

10572

In [17]:
sum(p.numel() for p in model.parameters())

21993280

In [18]:
model.head.weight.shape[0]

10572

In [44]:
optimizer = optim.Adam(model.parameters(), lr, betas = (momentum, 0.999), weight_decay = weight_decay)

### Checking

In [38]:
# for batch_index, (images, labels) in enumerate(train_loader):
#     invalid = (labels < 0) | (labels >= num_classes)

#     if invalid.any():
#         print("Invalid batch:", batch_index)
#         print("Invalid labels:", labels[invalid])
#         break

In [39]:
# for batch_index, (images, labels) in enumerate(val_loader):
#     invalid = (labels < 0) | (labels >= num_classes)

#     if invalid.any():
#         print("Invalid batch:", batch_index)
#         print("Invalid labels:", labels[invalid])
#         break

### Save Load Model

In [32]:
torch.save({
    "num_classes": num_classes,
    "model_state_dict": model.state_dict(),
}, "model.pth")

In [31]:
# checkpoint = torch.load("model.pth", map_location=device)

# model = Arcface(num_classes=checkpoint["num_classes"]).to(device)
# model.load_state_dict(checkpoint["model_state_dict"])

In [21]:
optimizer = optim.Adam(model.parameters(), lr, betas = (momentum, 0.999), weight_decay = weight_decay)

### Train

In [42]:
num_epochs = 1_000_000
num_repeat_per_batch = 3

In [23]:
len(train_loader)

12266

In [ ]:
avg_train_loss_list = []
avg_val_loss_list = []
avg_val_accuracy_list = []

for epoch in range(num_epochs):
    # Train
    model.train()
    train_loss = []
    
    print()
    print(f"Start Train - {epoch}")
    print()
    
    for batch_index, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        for _ in range(num_repeat_per_batch):
            optimizer.zero_grad()
            outputs = model.forward_train(images, labels)
            loss = nn.NLLLoss()(F.log_softmax(outputs, -1), labels)
            loss.backward()

            torch.nn.utils.clip_grad_value_(model.parameters(), clip_value=10.0)
            
            optimizer.step()
            train_loss.append(loss.item())
            
            print(f"   Epoch - {epoch} | Batch - {batch_index}: {loss.item()}")
            # break # Comment This
        
    avg_train_loss = sum(train_loss) / len(train_loss)
    avg_train_loss_list.append(avg_train_loss)
    print(f"Epoch - {epoch}: Train Loss - {avg_train_loss}")
    
    # Eval
    # model.eval()
    # val_loss = []
    # val_accuracy = []

    # print()
    # print(f"Start Eval - {epoch}")
    # print()
    
    # with torch.no_grad():
    #     for batch_index, (images, labels) in enumerate(train_loader):
    #         images, labels = images.to(device), labels.to(device)
            
    #         optimizer.zero_grad()
    #         outputs = model.forward_train(images, labels)
    #         loss = nn.NLLLoss()(F.log_softmax(outputs, -1), labels)   
    #         accuracy = torch.mean((torch.argmax(F.softmax(outputs, dim=-1), dim=-1) == labels).type(torch.FloatTensor))

    #         val_loss.append(loss.item())
    #         val_accuracy.append(accuracy.item())
            
    #         print(f"   Epoch - {epoch} | Batch - {batch_index}: Loss - {loss.item()} | Accuracy - {accuracy.item()}")
    #         # break # Comment This
            
    # avg_val_loss = sum(val_loss) / len(val_loss)
    # avg_val_accuracy = sum(val_accuracy) / len(val_accuracy)

    # avg_val_loss_list.append(avg_val_loss)
    # avg_val_accuracy_list.append(avg_val_accuracy)
    # print(f"Epoch - {epoch}: Val Loss - {avg_val_loss} | Val Accuracy - {avg_val_accuracy}")

    # break # Comment This


Start Train - 0

   Epoch - 0 | Batch - 0: 44.67536544799805
   Epoch - 0 | Batch - 0: 39.747352600097656
   Epoch - 0 | Batch - 0: 35.705936431884766
   Epoch - 0 | Batch - 1: 45.295623779296875
   Epoch - 0 | Batch - 1: 41.165557861328125
   Epoch - 0 | Batch - 1: 36.5977668762207
   Epoch - 0 | Batch - 2: 45.253517150878906
   Epoch - 0 | Batch - 2: 41.88935089111328
   Epoch - 0 | Batch - 2: 37.25736618041992
   Epoch - 0 | Batch - 3: 44.41838836669922
   Epoch - 0 | Batch - 3: 40.99121856689453
   Epoch - 0 | Batch - 3: 36.431915283203125
   Epoch - 0 | Batch - 4: 44.360557556152344
   Epoch - 0 | Batch - 4: 40.5459098815918
   Epoch - 0 | Batch - 4: 34.38861846923828
   Epoch - 0 | Batch - 5: 44.101036071777344
   Epoch - 0 | Batch - 5: 40.584251403808594
   Epoch - 0 | Batch - 5: 35.21682357788086
   Epoch - 0 | Batch - 6: 44.53514862060547
   Epoch - 0 | Batch - 6: 41.709312438964844
   Epoch - 0 | Batch - 6: 35.979736328125
   Epoch - 0 | Batch - 7: 44.407100677490234
   Epoc